# 🚀 CodeLens — Remote Repository Vector Indexer Pipeline

**Project**: CodeLens — Retrieval-Augmented Code Review Agent with Codebase-Aware Context  
**Compute Role**: Heavy Compute Offloading (Google Colab NVIDIA T4/A100 GPU)  
**Storage Role**: 5 TB Google Drive / Cloud Storage Tier (`codelens-storage`)

---

### 📌 Overview
This notebook offloads the heavy embedding and vector indexing compute from the local machine:
1. **Clones** the target GitHub repository (`TARGET_REPO`).
2. **Parses AST** using Tree-sitter for polyglot function and method extraction (Python, TypeScript, JavaScript).
3. **Generates Dense Vector Embeddings** using GPU-accelerated CodeBERT / SentenceTransformers.
4. **Builds a FAISS Index** (`IndexFlatIP` for inner product / cosine similarity).
5. **Exports Artifacts** (`faiss_index.bin` and `metadata.json`) directly to Google Drive or Google Cloud Storage.

In [ ]:
# Step 1: Install Required Dependencies
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gitpython", "tree-sitter", "tree-sitter-python", "tree-sitter-typescript", "faiss-cpu", "sentence-transformers", "google-cloud-storage"], check=True)


: 

In [ ]:
# Step 2: Pipeline Configuration
import os
from pathlib import Path

TARGET_REPO = "laxmi2577/code-review-agent"  # Change to your target repository
BRANCH = "main"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # or 'microsoft/codebert-base'
BATCH_SIZE = 64

OUTPUT_DIR = Path("./output_indices")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_EXPORT_PATH = "/content/drive/MyDrive/codelens-storage/indices"
print(f"Target Repository: {TARGET_REPO}")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")

In [ ]:
# Step 3: Mount Google Drive (Optional for 5TB Storage Export)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Skipping Google Drive mount (running outside Colab or rejected): {e}")

In [ ]:
# Step 4: Clone Repository
import shutil
from git import Repo

REPO_LOCAL_DIR = Path("./cloned_repo")
if REPO_LOCAL_DIR.exists():
    shutil.rmtree(REPO_LOCAL_DIR)

repo_url = f"https://github.com/{TARGET_REPO}.git"
print(f"Cloning {repo_url}...")
Repo.clone_from(repo_url, REPO_LOCAL_DIR, branch=BRANCH, depth=1)
print("Repository cloned successfully.")

In [ ]:
# Step 5: Polyglot AST Function Extraction with Tree-sitter
import tree_sitter_python
import tree_sitter_typescript
from tree_sitter import Language, Parser

py_lang = Language(tree_sitter_python.language())
ts_lang = Language(tree_sitter_typescript.language_typescript())

py_parser = Parser(py_lang)
ts_parser = Parser(ts_lang)

extracted_nodes = []

def extract_functions_from_tree(node, code_bytes, file_path, lang_name, class_name=None):
    nodes = []
    for child in node.children:
        current_class = class_name
        if child.type in ("class_definition", "class_declaration"):
            name_node = child.child_by_field_name("name")
            if name_node:
                current_class = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf-8", errors="ignore")

        if child.type in ("function_definition", "function_declaration", "method_definition"):
            name_node = child.child_by_field_name("name")
            fn_name = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf-8", errors="ignore") if name_node else "anonymous"
            snippet = code_bytes[child.start_byte:child.end_byte].decode("utf-8", errors="ignore")
            start_l = child.start_point[0] + 1
            end_l = child.end_point[0] + 1
            nodes.append({
                "file_path": str(file_path),
                "function_name": fn_name,
                "class_name": current_class,
                "language": lang_name,
                "start_line": start_l,
                "end_line": end_l,
                "code_snippet": snippet,
            })

        nodes.extend(extract_functions_from_tree(child, code_bytes, file_path, lang_name, current_class))
    return nodes

for root, dirs, files in os.walk(REPO_LOCAL_DIR):
    # Skip git and hidden directories
    if ".git" in dirs:
        dirs.remove(".git")
    for f in files:
        p = Path(root) / f
        rel_p = p.relative_to(REPO_LOCAL_DIR).as_posix()
        ext = p.suffix.lower()

        if ext == ".py":
            raw = p.read_bytes()
            tree = py_parser.parse(raw)
            extracted_nodes.extend(extract_functions_from_tree(tree.root_node, raw, rel_p, "python"))
        elif ext in (".ts", ".tsx", ".js", ".jsx"):
            raw = p.read_bytes()
            tree = ts_parser.parse(raw)
            extracted_nodes.extend(extract_functions_from_tree(tree.root_node, raw, rel_p, "typescript"))

print(f"Extracted {len(extracted_nodes)} function/method nodes across repository.")

In [ ]:
# Step 6: GPU Embedding Generation
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model {EMBEDDING_MODEL_NAME}...")
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

snippets = [n["code_snippet"] for n in extracted_nodes]
print(f"Computing embeddings for {len(snippets)} snippets...")

if snippets:
    embeddings = model.encode(
        snippets,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
else:
    # Fallback placeholder if empty repo
    dim = model.get_sentence_embedding_dimension()
    embeddings = np.empty((0, dim), dtype=np.float32)

print(f"Generated embeddings shape: {embeddings.shape}")

In [ ]:
# Step 7: Build FAISS Vector Index & Assemble Metadata
import json
import faiss

dimension = embeddings.shape[1] if embeddings.shape[0] > 0 else model.get_sentence_embedding_dimension()
index = faiss.IndexFlatIP(dimension)

if embeddings.shape[0] > 0:
    index.add(embeddings)

# Assign embedding_id corresponding to FAISS row
for idx, node in enumerate(extracted_nodes):
    node["embedding_id"] = str(idx)

metadata = {
    "repo_name": TARGET_REPO,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "dimension": dimension,
    "total_vectors": index.ntotal,
    "nodes": extracted_nodes,
}

index_path = OUTPUT_DIR / "faiss_index.bin"
metadata_path = OUTPUT_DIR / "metadata.json"

faiss.write_index(index, str(index_path))
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Successfully saved index snapshot to {index_path} ({index.ntotal} vectors)")
print(f"Successfully saved metadata snapshot to {metadata_path}")

In [ ]:
# Step 8: Export to 5TB Cloud Storage or Download Locally
safe_repo_name = TARGET_REPO.replace("/", "__").replace("\\", "__")

# 1. If Google Drive is mounted, copy files
drive_target = Path(DRIVE_EXPORT_PATH) / safe_repo_name
if Path("/content/drive/MyDrive").exists():
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(index_path, drive_target / "faiss_index.bin")
    shutil.copy2(metadata_path, drive_target / "metadata.json")
    print(f"Exported index to Google Drive: {drive_target}")

# 2. Direct browser download helper for Colab users
try:
    from google.colab import files
    print("Triggering browser download of index binaries for your local ./data/indices directory...")
    files.download(str(index_path))
    files.download(str(metadata_path))
except Exception:
    print("Local output available at:", OUTPUT_DIR.resolve())

In [ ]:
# Step 9: Sanity Retrieval Test
if index.ntotal > 0:
    test_query = "def login(user, password):"
    query_vec = model.encode([test_query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(query_vec, min(3, index.ntotal))

    print("--- Top 3 Retrieved Functions ---")
    for dist, idx in zip(distances[0], indices[0]):
        match = extracted_nodes[idx]
        print(f"Function: {match['function_name']} in {match['file_path']} (Score: {dist:.3f})")
else:
    print("Index is empty, skipping test search.")